## Data loading & pre-processing


In [1]:
import pandas as pd
import re

import analysis_utils as utils

Load data from all `../res` sub-directories (treatments):


In [2]:
summary_dfs = {}    # dict to store statistical summary data (<session_name>.json)
all_data_dfs = {}   # dict to store all individual measurements (all_data_<session_name>.json)

experiment_data_dir = '../res' # directory generated by "eval.py", contains the .json files with the experiment data

# Load all the experiment data
summary_dfs, all_data_dfs, raw_metric_data = utils.load_experiment_data(experiment_data_dir, iteration_structure=True)

### Flatten GPU and VRAM columns

These contains nested objects and values.
We are only interested in a subset of the contained data.


In [3]:
def flatten_gpu_vram_columns(df: pd.DataFrame) -> pd.DataFrame:
    # GPU columns
    df["gpu_util_mean"] = df["GPU"].apply(lambda x: x["utilization"]["avg"] if isinstance(x, dict) else None)
    df["gpu_util_max"]  = df["GPU"].apply(lambda x: x["utilization"]["max"] if isinstance(x, dict) else None)

    # VRAM columns
    df["vram_util_mean"]     = df["VRAM"].apply(lambda x: x["utilization"]["avg"] if isinstance(x, dict) else None)
    df["vram_util_max"]      = df["VRAM"].apply(lambda x: x["utilization"]["max"] if isinstance(x, dict) else None)
    df["vram_max_usage_mib"] = df["VRAM"].apply(lambda x: x["max_usage_MiB"]   if isinstance(x, dict) else None)

    # Drop the redundant columns
    df = df.drop(columns=["GPU", "VRAM"])

    return df

### Convert data to long format


In [4]:
def convert_dict_dfs_to_long(dictionary_dataframes: dict[pd.DataFrame], metrics_of_interest: list[str]) -> pd.DataFrame:
    rows = [] # temporary list to hold all constructed rows of data

    # Iterate through each treatment and corresponding data
    for treatment_name, df in dictionary_dataframes.items():

        # Extract the treatment factors (dataset is a blocked variable)
        model, quant, dataset = treatment_name.split("_")
        
        for i, row in df.iterrows():
            # Extract iteration from last dir in data_path (e.g. "01" from ".../01", etc.)
            match = re.search(r"/(\d{2})$", row["data_path"])
            iteration = int(match.group(1)) if match else i + 1 # FIXME: include the fallback or let it fail?

            # Construct row dictionary
            row_data = {
                "model": model, 
                "quantization": quant, 
                "dataset": dataset,
                "iteration": iteration
            }
        
            # Add relevant metrics
            for metric in metrics_of_interest:
                row_data[metric] = row[metric]

            rows.append(row_data)

    # Combine all the rows of data into a single long-format dataframe
    long_dataframe: pd.DataFrame = pd.DataFrame(rows)

    return long_dataframe

In [5]:
def convert_err_to_long(dictionary_dataframes: dict[pd.DataFrame], raw_metric_data: dict[str, dict],  metrics_of_interest: list[str]) -> pd.DataFrame:
    rows = [] # temporary list to hold all constructed rows of data

    # Iterate through each treatment and corresponding data
    for treatment_name, df in dictionary_dataframes.items():

        # Extract the treatment factors (dataset is a blocked variable)
        model, quant, dataset = treatment_name.split("_")
        
        # Construct row dictionary
        row_data = {
            "model": model, 
            "quantization": quant, 
            "dataset": dataset,
        }
        # Extract row for 'all_err' from summary df
        all_err_row = df[df.iloc[:, 0] == "all_err"]
        all_err_data = all_err_row.squeeze().to_dict() if not all_err_row.empty else {}

        # Get corresponding raw data (which includes the population array)
        raw_all_err_data = raw_metric_data.get(treatment_name, {}).get("all_err", {})

        # For each metric of interest, prefer raw data if available
        for metric in metrics_of_interest:
            if metric in raw_all_err_data:
                row_data[metric] = raw_all_err_data[metric]
            elif metric in all_err_data:
                row_data[metric] = all_err_data[metric]
            else:
                row_data[metric] = None  # fallback if missing

        rows.append(row_data)
    # Combine all the rows of data into a single long-format dataframe
    long_dataframe: pd.DataFrame = pd.DataFrame(rows)

    return long_dataframe

### Flatten columns and convert to long format to prepare for NHST analysis


In [6]:
# Metrics to extract (for RQ1)
RQ1_efficacy_metrics = [
    "accuracy", "recall", 
    "precision", "f1",
    "balanced_accuracy", 
    "specificity"  
]

# Metrics to extract (for RQ2)
RQ2_efficiency_metrics = [
    "time_to_analyze",
    "gpu_util_mean", "gpu_util_max",
    "vram_util_mean", "vram_util_max",
    "vram_max_usage_mib"
]

errors_metrics = [
    "mean",
    "sd",
    "total",
    "max",
    "population"
]

# Flatten and extract the relevant GPU and VRAM metrics into distinct columns (they are intially nested objects)
flattened_dfs = {
    key: flatten_gpu_vram_columns(df) for key, df in all_data_dfs.items()
}

# Convert data to long format; group by research question
efficacy_long_df = convert_dict_dfs_to_long(flattened_dfs, RQ1_efficacy_metrics)
efficiency_long_df = convert_dict_dfs_to_long(flattened_dfs, RQ2_efficiency_metrics)
errors_long_df = convert_err_to_long(summary_dfs, raw_metric_data, errors_metrics)

### Dump tidy/long format dataframes to CSV files


In [8]:
# NOTE: change the output filenames here
utils.save_dataframe_to_csv(efficacy_long_df, filename="rq1_flat_df-PT6", dest_path="./data/PT6_prompt")
utils.save_dataframe_to_csv(efficiency_long_df, filename="rq2_flat_df-PT6", dest_path="./data/PT6_prompt")
utils.save_dataframe_to_csv(errors_long_df, filename="err_summary_df-PT6", dest_path="./data/PT6_prompt")